# Phase 2 - Dataset setup

Unzip the Phase 2 dataset (downloaded from Zenodo) and point the kit at it. Run this **once**, before the other notebooks.

In [ ]:
import os, sys, warnings
from pathlib import Path
try:
    _here = Path(__vsc_ipynb_file__).resolve().parent   # VS Code sets this
except NameError:
    _here = Path.cwd().resolve()
_root = next(d for d in [_here, *_here.parents]
             if (d / 'part0_dataset_setup' / 'target_loader.py').exists())
os.chdir(_root)
sys.path[:0] = ['.', 'part0_dataset_setup', 'part1_forecast',
                'part2_siting', 'part3_economics']
warnings.filterwarnings('ignore')
import config

## 1. Locate the archive you downloaded

In [ ]:
import tarfile
# Path to the Phase 2 dataset archive you downloaded from Zenodo (edit if needed).
ARCHIVE = Path(os.environ.get('PHASE2_ARCHIVE', _root  / 'phase2_dataset.tgz')).expanduser()

# Folder to unpack into (needs ~12 GB free).
DEST = Path(os.environ.get('PHASE2_DEST', _root / 'phase2_data')).expanduser()
print('archive    :', ARCHIVE, '| exists:', ARCHIVE.exists())
print('unpack into:', DEST)

## 2. Unzip (skipped if the data is already in place)

In [ ]:
already = config.target_root().exists() and any(config.target_root().glob('*/arome_*.nc'))
if already:
    data_root = None
    print('target data already found at', config.target_root(), '- nothing to unzip')
elif ARCHIVE.exists():
    DEST.mkdir(parents=True, exist_ok=True)
    with tarfile.open(ARCHIVE) as tf:
        tf.extractall(DEST, filter='data')
    tops = [p for p in DEST.iterdir() if p.is_dir() and (p / 'train').exists()]
    data_root = tops[0] if tops else DEST
    print('extracted ->', data_root)
else:
    raise FileNotFoundError(
        f'Archive not found at {ARCHIVE}. Download the Phase 2 dataset from Zenodo and\n'
        f'set PHASE2_ARCHIVE to its path (or drop it at the default location above).')

## 3. Register the data root (so the other notebooks find it)

In [ ]:
if data_root is not None:
    (Path(config.__file__).parent / '.phase2_data_root').write_text(str(data_root.resolve()))
    os.environ['PHASE2_DATA_ROOT'] = str(data_root.resolve())
    import importlib; importlib.reload(config)
print(config.describe())

## 4. Sanity check - the data loads

In [ ]:
import target_loader, reanalysis_loader, splits
ad = target_loader.list_dates()
print(f'target days        : {len(ad):>5}  ({ad[0]} -> {ad[-1]})')
print(f'reanalysis days   : {len(reanalysis_loader.list_dates()):>5}')
st = target_loader.load_static()
print(f'target grid        : {st.lat.shape}  | sea cells: {int(st.sea.sum()):,}')
print(f'train years given : {splits.TRAIN_YEARS}')
print(f'hidden (no target) : {splits.EVAL_PUBLIC_YEAR} public, {splits.EVAL_FINAL_YEAR} final, {list(splits.SITING_YEARS)} siting')